# Notebook 01: BANKING77 Benchmark Data Exploration
### BankingLLM-Optimizer: Fine-Grained Banking Intent Detection

This notebook conducts comprehensive exploratory data analysis (EDA) on the **BANKING77** dataset:
1. Load canonical training and test splits
2. Inspect sample counts, missing values, and duplicate records
3. Analyze class distribution across all 77 fine-grained intents
4. Measure query character and token length distributions
5. Create a stratified 90/10 split on training data (9,002 dev pool, 1,001 validation)
6. Formally verify zero data leakage between development pool, validation, and untouched test splits
7. Export figures and dataset statistics tables


In [ ]:
# ==========================================
# 0. Google Colab / Local Environment Setup
# ==========================================
import sys, os
from pathlib import Path

# If running in Google Colab, install repository and dependencies
if "google.colab" in sys.modules:
    print("Detected Google Colab environment. Setting up...")
    !git clone https://github.com/your-username/banking-llm-optimizer.git
    %cd banking-llm-optimizer
    !pip install -r requirements.txt
    
    from google.colab import userdata
    try:
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
else:
    print("Running in local environment.")
    ROOT_DIR = Path(".").resolve()
    if str(ROOT_DIR) not in sys.path:
        sys.path.insert(0, str(ROOT_DIR))


### 1. Load Raw BANKING77 Dataset and Verify Schema


In [ ]:
from src.data.loader import BankingDataLoader
from src.data.preprocessing import TextPreprocessor
import pandas as pd

loader = BankingDataLoader()
train_raw, test_raw = loader.load_raw_data()

print(f"Raw Train shape: {train_raw.shape}")
print(f"Raw Test shape: {test_raw.shape}")
display(train_raw.head(5))


### 2. Data Quality: Missing Values, Duplicates, and Class Counts


In [ ]:
print("Missing values in Train:", train_raw.isnull().sum().to_dict())
print("Missing values in Test:", test_raw.isnull().sum().to_dict())

intents = loader.get_intent_labels()
print(f"Total Unique Intent Classes: {len(intents)}")
assert len(intents) == 77, "Expected exactly 77 intents."


### 3. Class Distribution and Query Length Analysis


In [ ]:
class_dist = TextPreprocessor.compute_class_distribution(train_raw)
print("Intent Distribution Summary (Top 5 & Bottom 5):")
display(pd.concat([class_dist.head(5), class_dist.tail(5)]))

TextPreprocessor.plot_class_distribution(train_raw, "results/figures/class_distribution.png")
TextPreprocessor.plot_text_length_distribution(train_raw, "results/figures/text_length_distribution.png")
print("Visualizations saved to results/figures/")


### 4. Create Stratified Train / Validation Split and Assert Zero Leakage


In [ ]:
train_pool, val_df = loader.create_stratified_split(train_raw, val_ratio=0.10, seed=42)

loader.verify_no_leakage(train_pool, val_df, test_raw)
loader.save_processed_splits(train_pool, val_df, test_raw)

print(f"Training / Few-Shot Example Pool: {len(train_pool)} examples")
print(f"Validation Set: {len(val_df)} examples")
print(f"Official Test Set: {len(test_raw)} examples")


### 5. Generate and Export Summary Statistics Table


In [ ]:
stats_data = {
    "Split": ["Training Pool (Dev)", "Validation Set", "Official Test Set", "Complete Dataset"],
    "Samples": [len(train_pool), len(val_df), len(test_raw), len(train_raw) + len(test_raw)],
    "Intents Represented": [train_pool["category"].nunique(), val_df["category"].nunique(), test_raw["category"].nunique(), len(intents)],
    "Mean Word Count": [
        round(TextPreprocessor.compute_text_length_stats(train_pool)["word_mean"], 2),
        round(TextPreprocessor.compute_text_length_stats(val_df)["word_mean"], 2),
        round(TextPreprocessor.compute_text_length_stats(test_raw)["word_mean"], 2),
        round(TextPreprocessor.compute_text_length_stats(pd.concat([train_raw, test_raw]))["word_mean"], 2),
    ],
    "P95 Word Count": [
        round(TextPreprocessor.compute_text_length_stats(train_pool)["word_p95"], 1),
        round(TextPreprocessor.compute_text_length_stats(val_df)["word_p95"], 1),
        round(TextPreprocessor.compute_text_length_stats(test_raw)["word_p95"], 1),
        round(TextPreprocessor.compute_text_length_stats(pd.concat([train_raw, test_raw]))["word_p95"], 1),
    ]
}
stats_df = pd.DataFrame(stats_data)
stats_df.to_csv("results/tables/dataset_statistics.csv", index=False)
display(stats_df)
print("Saved dataset statistics to results/tables/dataset_statistics.csv")
